# ⭐ Flujo 4: Modelo estrella
### Hechos + dimensiones

Unimos las ventas con sus
dimensiones para tener una
tabla ancha para analítica.

In [ ]:
from pyspark.sql import (
    SparkSession, functions as F)

spark = (
    SparkSession.builder
    .appName('modelo_estrella')
    .getOrCreate()
)

## 1️⃣ Tabla de HECHOS
### Solo IDs y métricas

Es la grande (millones).

In [ ]:
hechos = spark.createDataFrame([
  (1, 'P1', 'C1', 'T1', 120),
  (2, 'P2', 'C1', 'T2', 80),
  (3, 'P1', 'C2', 'T1', 200),
  (4, 'P3', 'C3', 'T2', 300),
], ['venta','id_prod','id_cli',
    'id_tienda','importe'])

hechos.show()

## 2️⃣ Las DIMENSIONES
### Tablas pequeñas

El detalle de cada ID.
Perfectas para broadcast.

In [ ]:
dim_prod = spark.createDataFrame([
  ('P1','Camiseta','Ropa'),
  ('P2','Café','Alimentación'),
  ('P3','Portátil','Electrónica'),
], ['id_prod','producto','categoria'])

dim_cli = spark.createDataFrame([
  ('C1','Ana','Madrid'),
  ('C2','Luis','Sevilla'),
  ('C3','Sara','Bilbao'),
], ['id_cli','cliente','ciudad'])

dim_tienda = spark.createDataFrame([
  ('T1','Online'),
  ('T2','Física'),
], ['id_tienda','canal'])

## 3️⃣ Encadenamos joins
### Broadcast de cada dim

La tabla de hechos no
se mueve por la red.

In [ ]:
ancha = (
  hechos
  .join(F.broadcast(dim_prod),
    'id_prod')
  .join(F.broadcast(dim_cli),
    'id_cli')
  .join(F.broadcast(dim_tienda),
    'id_tienda')
)

resultado = ancha.select(
  'venta', 'producto', 'categoria',
  'cliente', 'ciudad', 'canal',
  'importe')

resultado.show()

## 4️⃣ Analiza por lo que
## quieras
### Ya está todo unido

In [ ]:
(resultado
 .groupBy('canal')
 .agg(F.sum('importe').alias('total'))
 .show())

spark.stop()